# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [2]:
!pip -q install transformers sentencepiece torch


In [3]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [19]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [20]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [ ]:
# Task 1 — choose the most suitable model family for each task
answers = {
    "1. Sentiment classification":       "Encoder-only (e.g., BERT, DistilBERT)",
    "2. Machine translation":            "Encoder-decoder (e.g., T5, BART)",
    "3. Open-ended text generation":     "Decoder-only (e.g., GPT family)",
    "4. Named Entity Recognition":       "Encoder-only (e.g., BERT, RoBERTa)",
    "5. Text summarization":             "Encoder-decoder (e.g., T5, BART)",
}

for task, family in answers.items():
    print(f"{task:<32} -> {family}")

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [ ]:
# Task 2 — sentiment classification with the encoder-only model
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

for sentence in sentences:
    result = classifier(sentence)[0]
    print(f"Sentence : {sentence}")
    print(f"Label    : {result['label']}")
    print(f"Score    : {result['score']:.4f}")
    print()

## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [ ]:
# Task 3 — summarization with FLAN-T5 (encoder-decoder)
paragraph = (
    "Transformers are a type of neural network architecture that rely on self-attention "
    "to process sequences of text. Unlike recurrent networks, they can look at all words "
    "in a sentence at the same time, which makes training much faster and helps them "
    "capture long-range relationships between words. Transformers are the foundation of "
    "modern language models such as BERT, T5, and GPT, and they are used for tasks like "
    "translation, summarization, and question answering."
)

prompt = "Summarize the following paragraph in one sentence: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=60, num_beams=4)

print("Original paragraph:")
print(paragraph)
print("\nSummary:")
print(tokenizer.decode(output[0], skip_special_tokens=True))


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:**

A decoder-only model is trained to predict the next token from all previous tokens, which is exactly what a chatbot does: it reads the conversation so far and generates the reply one token at a time. Because the user's message and the model's own reply live in the same token sequence, the model can keep generating fluent, open-ended text and stay consistent with the earlier context. This single next-token objective also scales well to huge amounts of unlabeled text, giving the model broad knowledge and flexibility to follow instructions, answer questions, and hold a conversation without needing a separate encoder.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [ ]:
# Optional Challenge — sampling WITHOUT set_seed(42): run this cell several times
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

for i in range(3):
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.8, top_k=50)
    print(f"Run {i+1}: {out[0]['generated_text']}\n")

**Answer:**

With `do_sample=True`, the model does not always pick the single most likely next token. Instead, it randomly samples from the probability distribution over the vocabulary (shaped by settings like temperature and top-k). Because each token is drawn at random, small differences early on lead to different continuations, so the same prompt gives different outputs on each run. Calling `set_seed(42)` fixes the random number generator, making the results reproducible, and greedy decoding (`do_sample=False`) is always deterministic.